<a href="https://colab.research.google.com/github/pxs1990/NLP_LLM/blob/main/peft_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install -U transformers datasets peft accelerate scikit-learn pandas torch
! pip install -U bitsandbytes # Optional for low VRAM:

In [ ]:
import os, pandas as pd, numpy as np
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
import torch
from transformers import (
     AutoTokenizer,
     AutoModelForSequenceClassification,
     DataCollatorWithPadding,
     TrainingArguments,
     Trainer,
     )
from peft import LoraConfig, get_peft_model

In [ ]:
MODEL_NAME = "bert-base-uncased"  # any BERT-like model

In [ ]:
from sklearn.preprocessing import LabelEncoder

# ========= 1) Load & prepare data =========
df = pd.read_csv("train.csv")  # expects 'text','label'
assert {"text", "label"}.issubset(df.columns), "CSV must have text,label columns"

# --- Safety checks ---
if df["label"].isna().any():
    raise ValueError("Label column contains NaN values")

# ========= 2) Encode labels (0..N-1) =========
le = LabelEncoder()
df["label"] = le.fit_transform(df["label"])

# Build mappings (Hugging Face expects these)
label2id = {label: i for i, label in enumerate(le.classes_)}
id2label = {i: str(label) for label, i in label2id.items()}
num_labels = len(le.classes_)

# ========= 3) Train / validation split =========
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df["label"],
    random_state=42
)

# ========= 4) Convert to Hugging Face datasets =========
ds = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df, preserve_index=False),
})


In [ ]:
# ========= 2) Tokenizer =========
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def preprocess(row):
	return tokenizer(row["text"], truncation=True)
keep_cols = ["label"]
ds = ds.map(preprocess, batched=True, remove_columns=[c for c in ds["train"].column_names if c not in keep_cols])

collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:
# ========= 3) Base model =========
# (Optional low-VRAM: load in 8-bit with bitsandbytes; uncomment below)
# from transformers import BitsAndBytesConfig
# quant_cfg = BitsAndBytesConfig(load_in_8bit=True)
# base_model = AutoModelForSequenceClassification.from_pretrained(
#     MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id={v:k for k,v in id2label.items()},
#     quantization_config=quant_cfg, device_map="auto"
# )

base_model = AutoModelForSequenceClassification.from_pretrained(
	MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id={v: k for k, v in id2label.items()},
)

In [ ]:
# ========= 4) Apply LoRA adapters with PEFT =========
# For BERT, common target module substrings: "query","key","value","dense"
lora_cfg = LoraConfig(
	r=8,
    lora_alpha=16,
    target_modules=["query","key","value","dense"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS",
)
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()  # sanity check


In [ ]:
# ========= 5) Def Compute Metrics =========
def compute_metrics(p):
	preds = np.argmax(p.predictions, axis=1)
	return {
        "accuracy": accuracy_score(p.label_ids, preds),
        "f1_macro": f1_score(p.label_ids, preds, average="macro"),
	}


In [ ]:
# ========= 6) Def Training args & Trainer =========
args = TrainingArguments(
    output_dir="bert-lora-multiclass",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    learning_rate=2e-4,        	# LoRA can use a higher LR than full fine-tuning
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_strategy="steps",
    logging_steps=50,
    gradient_accumulation_steps=1,
    fp16=torch.cuda.is_available(),   # enable mixed precision if on GPU
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
	args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
# multi-levl Project structure
hier-bert/
  README.md
  requirements.txt
  configs/
    config.yaml
  data/
    train.csv
  src/
    train.py
    model.py
    data.py
    metrics.py
    infer.py
    utils.py

Data format (data/train.csv)

Required columns:

text (string)

level1 (parent label)

level2 (child label)

Example:

text,level1,level2
"Premier League match report",Sports,Football
"NBA finals analysis",Sports,Basketball
"New LLM model release",Tech,AI

requirements.txt
transformers>=4.40.0
datasets>=2.18.0
torch>=2.1.0
scikit-learn>=1.3.0
pandas>=2.0.0
pyyaml>=6.0.0
joblib>=1.3.0

configs/config.yaml
model_name: bert-base-uncased
max_length: 256

test_size: 0.1
random_state: 42

train:
  output_dir: outputs/hier_bert
  per_device_train_batch_size: 16
  per_device_eval_batch_size: 32
  learning_rate: 2.0e-5
  num_train_epochs: 3
  weight_decay: 0.01
  warmup_ratio: 0.06
  logging_steps: 50
  eval_strategy: epoch
  save_strategy: epoch
  load_best_model_at_end: true
  metric_for_best_model: eval_level2_f1_macro
  greater_is_better: true

loss_weights:
  level1: 1.0
  level2: 1.0

hierarchy:
  enforce_constraint_at_inference: true

src/utils.py
import json
from pathlib import Path
import joblib

def ensure_dir(path: str) -> None:
    Path(path).mkdir(parents=True, exist_ok=True)

def save_json(obj, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_joblib(obj, path: str) -> None:
    joblib.dump(obj, path)

def load_joblib(path: str):
    return joblib.load(path)

src/data.py
from dataclasses import dataclass
from typing import Dict, Tuple

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset, DatasetDict

@dataclass
class Encoders:
    level1: LabelEncoder
    level2: LabelEncoder

def build_hierarchy_map(df: pd.DataFrame, le_l1: LabelEncoder, le_l2: LabelEncoder) -> Dict[int, list]:
    """
    Returns: {level1_id: [allowed level2_id, ...], ...}
    """
    mapping = {}
    for l1, sub in df.groupby("level1"):
        l1_id = int(le_l1.transform([l1])[0])
        l2_ids = sorted(set(int(x) for x in le_l2.transform(sub["level2"].astype(str).tolist())))
        mapping[l1_id] = l2_ids
    return mapping

def load_and_prepare(
    csv_path: str,
    test_size: float,
    random_state: int
) -> Tuple[DatasetDict, Encoders, Dict[int, list]]:
    df = pd.read_csv(csv_path)
    required = {"text", "level1", "level2"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    # Safety checks
    if df[["text", "level1", "level2"]].isna().any().any():
        raise ValueError("Found NaN in text/level1/level2. Clean the data first.")

    # Fit encoders on full data (OK if you treat this as 'training universe').
    # If you want strictest evaluation, fit on train only, then transform val.
    le_l1 = LabelEncoder()
    le_l2 = LabelEncoder()

    df["level1_id"] = le_l1.fit_transform(df["level1"].astype(str))
    df["level2_id"] = le_l2.fit_transform(df["level2"].astype(str))

    hierarchy_map = build_hierarchy_map(df, le_l1, le_l2)

    # Stratify by the most specific label to keep distribution
    train_df, val_df = train_test_split(
        df,
        test_size=test_size,
        stratify=df["level2_id"],
        random_state=random_state
    )

    ds = DatasetDict({
        "train": Dataset.from_pandas(train_df[["text", "level1_id", "level2_id"]], preserve_index=False),
        "validation": Dataset.from_pandas(val_df[["text", "level1_id", "level2_id"]], preserve_index=False),
    })

    return ds, Encoders(level1=le_l1, level2=le_l2), hierarchy_map

src/model.py
from dataclasses import dataclass
from typing import Optional, Dict, Any

import torch
import torch.nn as nn
from transformers import AutoModel, PreTrainedModel, PretrainedConfig

@dataclass
class HierConfig:
    base_model_name: str
    num_level1: int
    num_level2: int
    dropout: float = 0.1

class HierBert(nn.Module):
    """
    BERT encoder + two classification heads.
    """
    def __init__(self, model_name: str, num_level1: int, num_level2: int, dropout: float = 0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.head_l1 = nn.Linear(hidden, num_level1)
        self.head_l2 = nn.Linear(hidden, num_level2)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None) -> Dict[str, torch.Tensor]:
        out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        # Use [CLS] pooled rep if available, else use first token
        if hasattr(out, "pooler_output") and out.pooler_output is not None:
            pooled = out.pooler_output
        else:
            pooled = out.last_hidden_state[:, 0]

        x = self.dropout(pooled)
        logits_l1 = self.head_l1(x)
        logits_l2 = self.head_l2(x)
        return {"logits_l1": logits_l1, "logits_l2": logits_l2}

src/metrics.py
from typing import Dict
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_level_metrics(y_true: np.ndarray, y_pred: np.ndarray, prefix: str) -> Dict[str, float]:
    return {
        f"{prefix}_acc": float(accuracy_score(y_true, y_pred)),
        f"{prefix}_f1_macro": float(f1_score(y_true, y_pred, average="macro")),
        f"{prefix}_f1_micro": float(f1_score(y_true, y_pred, average="micro")),
    }

src/train.py
import argparse
import yaml
import numpy as np
import torch
import torch.nn as nn

from transformers import AutoTokenizer, TrainingArguments, Trainer
from datasets import DatasetDict

from src.data import load_and_prepare
from src.model import HierBert
from src.metrics import compute_level_metrics
from src.utils import ensure_dir, save_joblib, save_json

class HierTrainer(Trainer):
    def __init__(self, *args, loss_w_l1=1.0, loss_w_l2=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_w_l1 = loss_w_l1
        self.loss_w_l2 = loss_w_l2
        self.ce = nn.CrossEntropyLoss()

    def compute_loss(self, model, inputs, return_outputs=False):
        level1 = inputs.pop("level1_id")
        level2 = inputs.pop("level2_id")

        outputs = model(**inputs)
        logits_l1 = outputs["logits_l1"]
        logits_l2 = outputs["logits_l2"]

        loss_l1 = self.ce(logits_l1, level1)
        loss_l2 = self.ce(logits_l2, level2)
        loss = self.loss_w_l1 * loss_l1 + self.loss_w_l2 * loss_l2

        if return_outputs:
            outputs["loss_l1"] = loss_l1.detach()
            outputs["loss_l2"] = loss_l2.detach()
            return loss, outputs
        return loss

def tokenize_batch(tokenizer, max_length: int, batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=max_length
    )

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", type=str, default="configs/config.yaml")
    ap.add_argument("--csv", type=str, default="data/train.csv")
    args = ap.parse_args()

    cfg = yaml.safe_load(open(args.config, "r", encoding="utf-8"))

    ds, encoders, hierarchy_map = load_and_prepare(
        csv_path=args.csv,
        test_size=float(cfg["test_size"]),
        random_state=int(cfg["random_state"]),
    )

    tokenizer = AutoTokenizer.from_pretrained(cfg["model_name"], use_fast=True)

    ds = ds.map(lambda b: tokenize_batch(tokenizer, int(cfg["max_length"]), b), batched=True)
    ds = ds.remove_columns(["text"])
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "level1_id", "level2_id"])

    num_l1 = len(encoders.level1.classes_)
    num_l2 = len(encoders.level2.classes_)

    model = HierBert(
        model_name=cfg["model_name"],
        num_level1=num_l1,
        num_level2=num_l2,
        dropout=0.1
    )

    out_dir = cfg["train"]["output_dir"]
    ensure_dir(out_dir)

    # Save encoders + hierarchy for inference
    save_joblib(encoders.level1, f"{out_dir}/labelenc_level1.joblib")
    save_joblib(encoders.level2, f"{out_dir}/labelenc_level2.joblib")
    save_json(hierarchy_map, f"{out_dir}/hierarchy_map.json")

    def compute_metrics(eval_pred):
        # eval_pred.predictions will be whatever Trainer returns; we need both heads.
        # We'll pack them from our custom prediction_step behavior by using a custom override:
        raise RuntimeError("compute_metrics is handled by custom evaluate loop below.")

    # --- Custom Trainer: override prediction_step to return both logits ---
    class _HierTrainer(HierTrainer):
        def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
            with torch.no_grad():
                level1 = inputs["level1_id"]
                level2 = inputs["level2_id"]
                outputs = model(
                    input_ids=inputs["input_ids"],
                    attention_mask=inputs.get("attention_mask", None),
                    token_type_ids=inputs.get("token_type_ids", None),
                )
                logits_l1 = outputs["logits_l1"]
                logits_l2 = outputs["logits_l2"]

                loss = None
                if not prediction_loss_only:
                    # Return logits in a tuple, Trainer will stack them
                    return loss, (logits_l1.detach().cpu().numpy(), logits_l2.detach().cpu().numpy()), (
                        level1.detach().cpu().numpy(), level2.detach().cpu().numpy()
                    )
                return loss, None, None

        def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval"):
            eval_dataset = eval_dataset if eval_dataset is not None else self.eval_dataset
            output = super().evaluate(eval_dataset=eval_dataset, ignore_keys=ignore_keys, metric_key_prefix=metric_key_prefix)
            return output

    training_args = TrainingArguments(
        output_dir=out_dir,
        per_device_train_batch_size=int(cfg["train"]["per_device_train_batch_size"]),
        per_device_eval_batch_size=int(cfg["train"]["per_device_eval_batch_size"]),
        learning_rate=float(cfg["train"]["learning_rate"]),
        num_train_epochs=float(cfg["train"]["num_train_epochs"]),
        weight_decay=float(cfg["train"]["weight_decay"]),
        warmup_ratio=float(cfg["train"]["warmup_ratio"]),
        logging_steps=int(cfg["train"]["logging_steps"]),
        eval_strategy=str(cfg["train"]["eval_strategy"]),
        save_strategy=str(cfg["train"]["save_strategy"]),
        load_best_model_at_end=bool(cfg["train"]["load_best_model_at_end"]),
        metric_for_best_model=str(cfg["train"]["metric_for_best_model"]),
        greater_is_better=bool(cfg["train"]["greater_is_better"]),
        fp16=torch.cuda.is_available(),
        report_to=[],
    )

    trainer = _HierTrainer(
        model=model,
        args=training_args,
        train_dataset=ds["train"],
        eval_dataset=ds["validation"],
        tokenizer=tokenizer,
        loss_w_l1=float(cfg["loss_weights"]["level1"]),
        loss_w_l2=float(cfg["loss_weights"]["level2"]),
    )

    trainer.train()

    # ---- Compute metrics after training (clean + explicit) ----
    pred = trainer.predict(ds["validation"])
    # pred.predictions is tuple: (logits_l1, logits_l2)
    logits_l1, logits_l2 = pred.predictions
    y1_true, y2_true = pred.label_ids

    y1_pred = np.argmax(logits_l1, axis=1)
    y2_pred = np.argmax(logits_l2, axis=1)

    metrics = {}
    metrics.update(compute_level_metrics(y1_true, y1_pred, "level1"))
    metrics.update(compute_level_metrics(y2_true, y2_pred, "level2"))

    # Save final metrics
    save_json(metrics, f"{out_dir}/final_metrics.json")
    print("Final metrics:", metrics)

    # Save model + tokenizer
    trainer.save_model(out_dir)
    tokenizer.save_pretrained(out_dir)

if __name__ == "__main__":
    main()

src/infer.py (with hierarchy constraint)
import argparse
import numpy as np
import torch
from transformers import AutoTokenizer
from src.model import HierBert
from src.utils import load_joblib, load_json

@torch.no_grad()
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model_dir", type=str, required=True)
    ap.add_argument("--text", type=str, required=True)
    args = ap.parse_args()

    le_l1 = load_joblib(f"{args.model_dir}/labelenc_level1.joblib")
    le_l2 = load_joblib(f"{args.model_dir}/labelenc_level2.joblib")
    hierarchy_map = load_json(f"{args.model_dir}/hierarchy_map.json")

    tokenizer = AutoTokenizer.from_pretrained(args.model_dir, use_fast=True)

    num_l1 = len(le_l1.classes_)
    num_l2 = len(le_l2.classes_)

    model = HierBert(model_name=args.model_dir, num_level1=num_l1, num_level2=num_l2)
    model.load_state_dict(torch.load(f"{args.model_dir}/pytorch_model.bin", map_location="cpu"))
    model.eval()

    batch = tokenizer(args.text, return_tensors="pt", truncation=True, padding=True, max_length=256)
    out = model(**batch)

    logits_l1 = out["logits_l1"].cpu().numpy()[0]
    logits_l2 = out["logits_l2"].cpu().numpy()[0]

    l1_id = int(np.argmax(logits_l1))
    l1_label = str(le_l1.inverse_transform([l1_id])[0])

    # Optional constraint: allow only children under predicted l1
    allowed = hierarchy_map.get(str(l1_id), hierarchy_map.get(l1_id, None))
    if allowed is not None:
        mask = np.full_like(logits_l2, -1e9, dtype=np.float32)
        for cid in allowed:
            mask[int(cid)] = 0.0
        logits_l2 = logits_l2 + mask

    l2_id = int(np.argmax(logits_l2))
    l2_label = str(le_l2.inverse_transform([l2_id])[0])

    print({"level1_id": l1_id, "level1": l1_label, "level2_id": l2_id, "level2": l2_label})

if __name__ == "__main__":
    main()

# **QA-RAG Folder structure**
rag_local_llama/

  .env.example

  .gitignore

  requirements.txt

  data/

    Resume_Pushpa_Shrestha.pdf

    index/                 # created after build

  src/

    rag_local_llama/

      __init__.py

      config.py

      pdf_loader.py

      chunker.py

      embedder.py

      faiss_store.py

      llm_hf.py

      rag_pipeline.py
      
      cli.py

In [ ]:


# 2) requirements.txt
python-dotenv
pypdf
faiss-cpu
numpy>=1.25,<3.0

transformers>=4.44
accelerate
sentencepiece
torch

langchain-core>=0.3.72,<1.0.0
langchain-community>=0.3.9,<0.4.0
langchain-text-splitters>=0.3.9,<1.0.0


# Install:

# pip install -r requirements.txt

# 3) .env

# Optional (needed only for gated models on HF)
HF_TOKEN=

PDF_PATH=./data/Resume_Pushpa_Shrestha.pdf
INDEX_DIR=./data/index

# LLM for generation (choose one that fits your GPU/CPU)
LLM_ID=LLM_ID=TinyLlama/TinyLlama-1.1B-Chat-v1.0

# Embeddings model
EMB_ID=sentence-transformers/all-MiniLM-L6-v2


# src/rag_local_llama/config.py
from dataclasses import dataclass
import os
from dotenv import load_dotenv

load_dotenv()

@dataclass(frozen=True)
class RAGConfig:
    pdf_path: str = os.getenv("PDF_PATH", "./data/Resume_Pushpa_Shrestha.pdf")
    index_dir: str = os.getenv("INDEX_DIR", "./data/index")

    top_k: int = 5
    chunk_size: int = 800
    chunk_overlap: int = 150

    emb_model_id: str = os.getenv("EMB_ID", "sentence-transformers/all-MiniLM-L6-v2")
    llm_model_id: str = os.getenv("LLM_ID", "NousResearch/Hermes-3-Llama-3.1-8B")

    max_new_tokens: int = 256
    temperature: float = 0.2

# src/rag_local_llama/pdf_loader.py
from typing import List
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.docstore.document import Document

class PDFDocLoader:
    def __init__(self, pdf_path: str):
        self.pdf_path = pdf_path

    def load(self) -> List[Document]:
        return PyPDFLoader(self.pdf_path).load()

# src/rag_local_llama/chunker.py
from typing import List
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.docstore.document import Document

class TextChunker:
    def __init__(self, chunk_size: int = 800, chunk_overlap: int = 150):
        self.textsplitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )

    def split(self, docs: List[Document]) -> List[Document]:
        return self.textsplitter.split_documents(docs)

# src/rag_local_llama/embedder.py
from typing import List
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

class MiniLMEmbedder:
    """
    HF AutoModel mean pooling + attention mask, L2-normalized.
    """
    def __init__(self, model_id: str):
        self.tok = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModel.from_pretrained(model_id)
        self.model.eval()

    @torch.no_grad()
    def embed(self, texts: List[str]) -> np.ndarray:
        batch = self.tok(texts, return_tensors="pt", padding=True, truncation=True)
        out = self.model(**batch)
        last_hidden = out.last_hidden_state           # [B, T, H]
        mask = batch["attention_mask"].unsqueeze(-1)  # [B, T, 1]

        summed = (last_hidden * mask).sum(dim=1)      # [B, H]
        counts = mask.sum(dim=1).clamp(min=1)         # [B, 1]
        embs = (summed / counts).cpu().numpy()

        embs = embs / (np.linalg.norm(embs, axis=1, keepdims=True) + 1e-12)
        return embs.astype("float32")

# src/rag_local_llama/faiss_store.py
from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, List
import os, pickle
import numpy as np
import faiss

from langchain_community.docstore import InMemoryDocstore
from langchain_community.docstore.document import Document

@dataclass
class RetrievedChunk:
    doc: Document
    score: float
    idx: int

class FAISSStore:
    """
    Cosine similarity retrieval using IndexFlatIP on normalized vectors.
    Persists FAISS index + docstore + mapping.
    """
    def __init__(self, dim: int):
        self.index = faiss.IndexFlatIP(dim)
        self.docstore = InMemoryDocstore()
        self.index_to_doc: Dict[int, str] = {}
        self._next_id = 0

    def add(self, docs: List[Document], vectors: np.ndarray) -> None:
        assert len(docs) == vectors.shape[0]
        for doc, v in zip(docs, vectors):
            self.index.add(v[np.newaxis, :])
            key = f"doc-{self._next_id}"
            self.docstore.add({key: doc})
            self.index_to_doc[self._next_id] = key
            self._next_id += 1

    def search(self, query_vec: np.ndarray, k: int = 5) -> List[RetrievedChunk]:
        D, I = self.index.search(query_vec, k)
        out: List[RetrievedChunk] = []
        for score, idx in zip(D[0], I[0]):
            if idx == -1:
                continue
            key = self.index_to_doc.get(int(idx))
            if not key:
                continue
            doc = self.docstore.search(key)
            out.append(RetrievedChunk(doc=doc, score=float(score), idx=int(idx)))
        return out

    def save(self, folder: str) -> None:
        os.makedirs(folder, exist_ok=True)
        faiss.write_index(self.index, os.path.join(folder, "faiss.index"))
        with open(os.path.join(folder, "docstore.pkl"), "wb") as f:
            pickle.dump(self.docstore, f)
        with open(os.path.join(folder, "mapping.pkl"), "wb") as f:
            pickle.dump({"index_to_doc": self.index_to_doc, "_next_id": self._next_id}, f)

    @classmethod
    def load(cls, folder: str) -> "FAISSStore":
        index = faiss.read_index(os.path.join(folder, "faiss.index"))
        store = cls(dim=index.d)
        store.index = index
        with open(os.path.join(folder, "docstore.pkl"), "rb") as f:
            store.docstore = pickle.load(f)
        with open(os.path.join(folder, "mapping.pkl"), "rb") as f:
            d = pickle.load(f)
            store.index_to_doc = d["index_to_doc"]
            store._next_id = d["_next_id"]
        return store

# src/rag_local_llama/llm_hf.py
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

class HFCausalLM:
    def __init__(self, model_id: str):
        token = os.getenv("HF_TOKEN") or None

        self.tok = AutoTokenizer.from_pretrained(model_id, use_fast=True, token=token)

        dtype = (
            torch.bfloat16
            if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
            else torch.float16
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=dtype,
            device_map="auto",
            token=token,
        )

        # Some models need explicit pad token
        if self.tok.pad_token_id is None and self.tok.eos_token_id is not None:
            self.tok.pad_token = self.tok.eos_token

    def generate(self, system_prompt: str, user_prompt: str,
                 max_new_tokens: int = 256, temperature: float = 0.2) -> str:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]

        if hasattr(self.tok, "apply_chat_template") and self.tok.chat_template:
            prompt = self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        else:
            prompt = f"{system_prompt}\n\n{user_prompt}"

        inputs = self.tok(prompt, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            out_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=True if temperature > 0 else False,
                pad_token_id=self.tok.pad_token_id,
            )

        text = self.tok.decode(out_ids[0], skip_special_tokens=True)

        # Try to remove the prompt portion
        if user_prompt in text:
            return text.split(user_prompt, 1)[-1].strip()
        return text.strip()

# src/rag_local_llama/rag_pipeline.py
from typing import Any, Dict
from .pdf_loader import PDFDocLoader
from .chunker import TextChunker
from .embedder import MiniLMEmbedder
from .faiss_store import FAISSStore
from .llm_hf import HFCausalLM

class RAGPipeline:
    def __init__(self, pdf_path: str, index_dir: str,
                 emb_model_id: str, llm_model_id: str,
                 chunk_size: int = 800, chunk_overlap: int = 150):
        self.pdf_path = pdf_path
        self.index_dir = index_dir

        self.loader = PDFDocLoader(pdf_path)
        self.chunker = TextChunker(chunk_size, chunk_overlap)
        self.embedder = MiniLMEmbedder(emb_model_id)
        self.llm = HFCausalLM(llm_model_id)

        self.store: FAISSStore | None = None

    def build_index(self) -> None:
        raw_docs = self.loader.load()
        chunks = self.chunker.split(raw_docs)
        texts = [d.page_content for d in chunks]
        vecs = self.embedder.embed(texts)

        store = FAISSStore(dim=vecs.shape[1])
        store.add(chunks, vecs)
        store.save(self.index_dir)
        self.store = store

    def load_index(self) -> None:
        self.store = FAISSStore.load(self.index_dir)

    def retrieve(self, query: str, k: int = 5):
        if self.store is None:
            self.load_index()
        qv = self.embedder.embed([query])
        return self.store.search(qv, k=k)

    def answer(self, question: str, k: int = 5, return_sources: bool = True,
               max_new_tokens: int = 256, temperature: float = 0.2) -> Dict[str, Any]:
        hits = self.retrieve(question, k=k)
        context = "\n\n".join([h.doc.page_content for h in hits])

        system = (
            "You are an assistant that answers STRICTLY from the provided résumé context. "
            "If the answer is not present, say exactly: 'I cannot find that in the résumé.' "
            "Do not invent dates, ages, or employers."
        )
        user = f"Résumé context:\n{context}\n\nQuestion: {question}\nAnswer concisely:"

        ans = self.llm.generate(
            system, user,
            max_new_tokens=max_new_tokens,
            temperature=temperature
        )

        out: Dict[str, Any] = {"answer": ans}
        if return_sources:
            out["sources"] = [
                {
                    "score": round(h.score, 4),
                    "meta": h.doc.metadata,
                    "preview": (h.doc.page_content[:240].replace("\n", " ")
                                + ("..." if len(h.doc.page_content) > 240 else "")),
                }
                for h in hits
            ]
        return out

# src/rag_local_llama/cli.py
import argparse, json
from .config import RAGConfig
from .rag_pipeline import RAGPipeline

def main():
    cfg = RAGConfig()

    p = argparse.ArgumentParser()
    p.add_argument("--build", action="store_true", help="Build/Rebuild FAISS index")
    p.add_argument("--q", type=str, help="Ask a question")
    p.add_argument("--k", type=int, default=cfg.top_k, help="Top-K retrieval")
    args = p.parse_args()

    rag = RAGPipeline(
        pdf_path=cfg.pdf_path,
        index_dir=cfg.index_dir,
        emb_model_id=cfg.emb_model_id,
        llm_model_id=cfg.llm_model_id,
        chunk_size=cfg.chunk_size,
        chunk_overlap=cfg.chunk_overlap,
    )

    if args.build:
        rag.build_index()
        print(f"✅ Index built at: {cfg.index_dir}")

    if args.q:
        result = rag.answer(
            args.q, k=args.k, return_sources=True,
            max_new_tokens=cfg.max_new_tokens,
            temperature=cfg.temperature,
        )
        print(json.dumps(result, indent=2))

if __name__ == "__main__":
    main()

# **QA - RAG Project (Local LLM + text data)**
# **📁 Project Structure**

QA_rag_project/

├── data/

    ├── docs.txt                 # your corpus

    └── index/                   # FAISS index

├── models/

    └── phi-3-mini.gguf          # local LLM

├── src/

    ├── config.py

    ├── loader.py

    ├── embedder.py

    ├── vector_store.py

    ├── llm.py

    ├── rag.py

    └── main.py

├── requirements.txt

└── README.md

In [ ]:
#  requirements.txt
numpy
faiss-cpu
sentence-transformers
torch
llama-cpp-python

In [ ]:
# ⚙️ config.py
from dataclasses import dataclass

@dataclass(frozen=True)
class RAGConfig:
    docs_path: str = "./data/docs.txt"
    index_path: str = "./data/index"

    embedding_model: str = "BAAI/bge-small-en-v1.5"

    llm_path: str = "./models/phi-3-mini.gguf"
    n_ctx: int = 2048
    max_tokens: int = 300
    temperature: float = 0.2

    top_k: int = 3

In [ ]:
# 📄 loader.py (document loader)
from typing import List

class TextLoader:
    def __init__(self, path: str):
        self.path = path

    def load(self) -> List[str]:
        with open(self.path, "r", encoding="utf-8") as f:
            text = f.read()
        return [t.strip() for t in text.split(".") if t.strip()]

In [ ]:
# 🔢 embedder.py
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import List

class Embedder:
    def __init__(self, model_name: str):
        self.model = SentenceTransformer(model_name)

    def embed(self, texts: List[str]) -> np.ndarray:
        return self.model.encode(
            texts,
            normalize_embeddings=True,
            convert_to_numpy=True
        ).astype("float32")

In [ ]:
# 🧠 vector_store.py (FAISS)
import faiss
import numpy as np
from typing import List

class VectorStore:
    def __init__(self, dim: int):
        self.index = faiss.IndexFlatIP(dim)
        self.texts = []

    def add(self, embeddings: np.ndarray, texts: List[str]):
        self.index.add(embeddings)
        self.texts.extend(texts)

    def search(self, query_emb: np.ndarray, k: int):
        scores, idxs = self.index.search(query_emb, k)
        return [self.texts[i] for i in idxs[0]]

In [ ]:
# 🤖 llm.py (local LLM via llama.cpp)
from llama_cpp import Llama

class LocalLLM:
    def __init__(self, model_path: str, n_ctx: int):
        self.llm = Llama(
            model_path=model_path,
            n_ctx=n_ctx,
            n_gpu_layers=-1,
            verbose=False
        )

    def generate(self, prompt: str, max_tokens: int, temperature: float) -> str:
        out = self.llm(
            prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            stop=["</assistant>"]
        )
        return out["choices"][0]["text"].strip()

In [ ]:
# 🔗 rag.py (RAG pipeline)
class RAGPipeline:
    def __init__(self, embedder, vector_store, llm, config):
        self.embedder = embedder
        self.store = vector_store
        self.llm = llm
        self.config = config

    def ask(self, question: str) -> str:
        q_emb = self.embedder.embed([question])
        context = self.store.search(q_emb, self.config.top_k)

        prompt = f"""
            You are a helpful assistant. Answer ONLY using the context.

            Context:
            {chr(10).join(context)}

            Question: {question}
            Answer:
            """
        return self.llm.generate(
            prompt,
            max_tokens=self.config.max_tokens,
            temperature=self.config.temperature
        )

In [ ]:
# 🚀 main.py (run everything)
from config import RAGConfig
from loader import TextLoader
from embedder import Embedder
from vector_store import VectorStore
from llm import LocalLLM
from rag import RAGPipeline

cfg = RAGConfig()

# Load documents
texts = TextLoader(cfg.docs_path).load()

# Embed
embedder = Embedder(cfg.embedding_model)
embeddings = embedder.embed(texts)

# Vector store
store = VectorStore(dim=embeddings.shape[1])
store.add(embeddings, texts)

# LLM
llm = LocalLLM(cfg.llm_path, cfg.n_ctx)

# RAG
rag = RAGPipeline(embedder, store, llm, cfg)

print(rag.ask("Income generated"))